In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [12]:
base_url = 'https://raw.githubusercontent.com/londonaicentre/datatakehome/refs/heads/main/data'
encounters = pd.read_csv(os.path.join(base_url,'encounters.csv'))
encounters2 = pd.read_csv(os.path.join(base_url,'encounters_schema_change_batch.csv'))

# Data Quality Issues - encounters.csv

## 1. Presence of encounters with a start date after end date

In [42]:
# convert datetime columns to datetime
encounters['START'] = pd.to_datetime(encounters['START'])
encounters['STOP'] = pd.to_datetime(encounters['STOP'])

#subset of enc with start date > end date
enc_start_after_end = encounters[encounters['START'] > encounters['STOP']]
print(f'% of encounters with start date after end date: {(enc_start_after_end.shape[0] / encounters.shape[0])*100} %')

#create ed encounter subset
encounters_ed = encounters[encounters['ENCOUNTERCLASS'].isin(['emergency'])]

#subset of ed enc with start > end
enc_ed_start_after_end = encounters_ed[encounters_ed['START'] > encounters_ed['STOP']]
print(f'% of ED encounters with start date after end date: {(enc_ed_start_after_end.shape[0] / encounters_ed.shape[0])*100} %')

% of encounters with start date after end date: 1.9547942124597046 %
% of ED encounters with start date after end date: 1.7224880382775118 %


In [43]:
print(f'Examples of DQ rows :')
enc_ed_start_after_end[['Id','START','STOP']].head(2)

Examples of DQ rows :


,Id,START,STOP
4012,bf72df79-0d4b-463c-99e2-cfbab0a2609b,2015-03-05 01:32:46+00:00,2015-02-25 23:32:46+00:00
4155,be018dec-1687-43d7-9504-704520d6e2cf,2013-12-12 15:00:35+00:00,2013-12-04 13:00:35+00:00


## 2. Presence of encounters with very long LOS that is clinically implausible

In [53]:
encounters['LOS_Days'] = (encounters['STOP'] - encounters['START']) / np.timedelta64(1, 'D')

long_los = encounters[encounters['LOS_Days'] > 5][['Id', 'START', 'STOP']]

print(f'number of encounters with LOS > 5 days = {long_los.shape[0]}, {(long_los.shape[0]/encounters.shape[0])*100} %')
long_los.head()

number of encounters with LOS > 5 days = 1363, 2.5545393207886646 %


,Id,START,STOP
16,d3bb3a95-fd24-415b-9675-39bf616e44b3,1884-07-09 00:36:00+00:00,2012-08-12 00:41:23+00:00
59,e032c21a-5a60-48ba-8357-63526d8ab4a7,1826-10-06 20:53:00+00:00,2011-12-10 06:36:56+00:00
81,7ac493f1-fb2b-4bc9-9583-82ec321992a8,1844-09-09 19:34:00+00:00,2019-01-04 03:23:42+00:00
87,47a62056-2f39-4c39-a3bd-f12069d65a5e,1821-08-05 22:47:00+00:00,2019-06-19 06:42:24+00:00
92,c855eddb-1301-493b-b1b1-7f5c3c5a934f,1822-02-20 12:39:00+00:00,2004-07-31 16:05:36+00:00


## 3. Presence of encounters with missing reason code / description

In [7]:
# all encounters
enc_count = encounters.shape[0]
enc_missing_reason_count = encounters[encounters["REASONDESCRIPTION"].isna() | encounters["REASONCODE"].isna()].shape[0]

print(f'Total encounters: {enc_count}')
print(f'Total encounters with missing reason code/desc: {enc_missing_reason_count}')
print(f'Missing reason rate: {enc_missing_reason_count / enc_count}\n')

# ed subset
encounters_ed = encounters[encounters['ENCOUNTERCLASS'].isin(['emergency'])]
ed_enc_count = encounters_ed.shape[0]
ed_enc_missing_reason_count = encounters_ed[encounters_ed["REASONDESCRIPTION"].isna() | encounters_ed["REASONCODE"].isna()].shape[0]

print(f'Total ED encounters: {ed_enc_count}')
print(f'Total ED encounters with missing reason code/desc: {ed_enc_missing_reason_count}')
print(f'ED missing reason rate: {ed_enc_missing_reason_count / ed_enc_count}')

Total encounters: 53356
Total encounters with missing reason code/desc: 39578
Missing reason rate: 0.7417722467951121

Total ED encounters: 2090
Total ED encounters with missing reason code/desc: 1350
ED missing reason rate: 0.645933014354067


## 4. Encounter code can be mapped to multiple descriptions

In [ ]:
enc_desc = encounters[['CODE','DESCRIPTION']].dropna().drop_duplicates()
code_desc_match_ind = enc_desc['CODE'].nunique() == enc_desc['DESCRIPTION'].nunique()
print(f'Enc Code-Desc 1:1 Match: {code_desc_match_ind}')
enc_desc.groupby(by='CODE').agg('count').sort_values(by='DESCRIPTION', ascending=False).head()


Enc Code-Desc 1:1 Match: False


,DESCRIPTION
CODE,
162673000,5
50849002,3
185349003,3
185347001,3
185345009,3


In [9]:
# example
enc_desc[enc_desc['CODE'] == 162673000]

,CODE,DESCRIPTION
1,162673000,General examination of patient (procedure)
53347,162673000,Orphan encounter 2
53349,162673000,Orphan encounter 4
53354,162673000,Orphan encounter 9
53355,162673000,Orphan encounter 10


In [10]:
#ed encounters
ed_enc_desc = encounters_ed[['CODE','DESCRIPTION']].dropna().drop_duplicates()
ed_code_desc_match_ind = enc_desc['CODE'].nunique() == ed_enc_desc['DESCRIPTION'].nunique()
print(f'ED Enc Code-Desc 1:1 Match: {ed_code_desc_match_ind}')
ed_enc_desc.groupby(by='CODE').agg('count').sort_values(by='DESCRIPTION', ascending=False).head()

ED Enc Code-Desc 1:1 Match: False


,DESCRIPTION
CODE,
50849002,3
22298006,1
32485007,1
183460006,1
183478001,1


In [11]:
# not observed with reasoncodes and reasondescription
enc_visit_reasons = encounters[['REASONCODE','REASONDESCRIPTION']].dropna().drop_duplicates()
enc_visit_code_desc_match_ind = enc_visit_reasons['REASONCODE'].nunique() == enc_visit_reasons['REASONDESCRIPTION'].nunique()
enc_visit_reasons['REASONCODE'].nunique() == enc_visit_reasons['REASONDESCRIPTION'].nunique()
print(f'ED Enc Reason Code-Desc 1:1 Match: {enc_visit_code_desc_match_ind}')

ED Enc Reason Code-Desc 1:1 Match: True


In [ ]:
#schema change table
encounters2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   200 non-null    object 
 1   PATIENT              200 non-null    object 
 2   ENCOUNTER_TYPE       200 non-null    object 
 3   CODE                 200 non-null    int64  
 4   DESCRIPTION          200 non-null    object 
 5   START                200 non-null    int64  
 6   STOP                 188 non-null    float64
 7   ORGANIZATION         200 non-null    object 
 8   PROVIDER             200 non-null    object 
 9   PAYER                200 non-null    object 
 10  BASE_ENCOUNTER_COST  200 non-null    float64
 11  TOTAL_CLAIM_COST     200 non-null    float64
 12  PAYER_COVERAGE       200 non-null    float64
 13  REASONCODE           45 non-null     float64
 14  REASONDESCRIPTION    45 non-null     object 
 15  SOURCE_SYSTEM        200 non-null    obj

In [ ]:
encounter2_ed = encounters2[encounters2['ENCOUNTER_TYPE']== 'emergency']
encounter2_ed['REASONCODE'] = encounter2_ed['REASONCODE'].astype('Int64')
encounter2_ed['START'] = pd.to_datetime(encounter2_ed['START'], origin = 'unix', unit='ms')
encounter2_ed['STOP'] = pd.to_datetime(encounter2_ed['STOP'], origin = 'unix', unit='ms')
encounter2_ed.head()

/var/folders/51/1bfgt4ls78d2ckf9flwx16jm0000gn/T/ipykernel_76658/1022806386.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  encounter2_ed['REASONCODE'] = encounter2_ed['REASONCODE'].astype('Int64')
/var/folders/51/1bfgt4ls78d2ckf9flwx16jm0000gn/T/ipykernel_76658/1022806386.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  encounter2_ed['START'] = pd.to_datetime(encounter2_ed['START'], origin = 'unix', unit='ms')
/var/folders/51/1bfgt4ls78d2ckf9flwx16jm0000gn/T/ipykernel_76658/1022806386.py:5: SettingW

,Id,PATIENT,ENCOUNTER_TYPE,CODE,DESCRIPTION,START,STOP,ORGANIZATION,PROVIDER,PAYER,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION,SOURCE_SYSTEM
76,27b3acba-4394-409a-a022-3b79da7a8f84,a708bca4-8fc9-4567-8c22-8b13d44f4f45,emergency,50849002,Emergency Encounter,2019-01-25 04:27:38,2019-01-25 05:42:38,d733d4a9-080d-3593-b910-2366e652b7ea,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,7caa7254-5050-3b5e-9eae-bd5ea30e809c,129.16,129.16,89.16,<NA>,NaN,CERNER_LEGACY
77,267f25e6-82fc-4966-9ce7-ab1632a68523,875d9a6a-1c7e-44a9-96a0-b4ba92e40841,emergency,50849002,Emergency room admission (procedure),2012-07-14 03:44:13,2012-07-14 04:44:13,3d10019f-c88e-3de5-9916-6107b9c0263d,4b04cd2f-3f27-35bc-8069-f4ca6339529f,4d71f845-a6a9-3c39-b242-14d25ef86a8d,129.16,129.16,64.16,<NA>,NaN,EPIC_PROD
123,3144dfd1-037f-416c-bf3b-96d02668203a,f504b982-e99b-4064-ad25-9e5480e769cd,emergency,50849002,Emergency room admission (procedure),2011-09-15 23:16:26,2011-09-16 00:16:26,5d4b9df1-93ae-3bc9-b680-03249990e558,af01a385-31d3-3c77-8fdb-2867fe88df2f,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,129.16,129.16,69.16,<NA>,NaN,EPIC_PROD
125,2143f391-595c-4527-9b3d-68df74e28d26,53d371cc-eb68-4740-a68c-39707c6cb519,emergency,50849002,Emergency room admission (procedure),2012-04-30 17:12:03,2012-04-30 19:31:03,465de31f-3098-365c-af70-48a071e1f5aa,0a8a9359-7b33-3256-a068-b5a7d18ebe4b,7caa7254-5050-3b5e-9eae-bd5ea30e809c,129.16,129.16,89.16,<NA>,NaN,EPIC_PROD
145,9a1650bf-5c2e-472d-a8fd-eb264321167c,5bb2161d-1383-408b-a96d-344c0bd5f23a,emergency,183460006,Obstetric emergency hospital admission,2018-12-11 06:15:56,2018-12-11 07:45:56,d733d4a9-080d-3593-b910-2366e652b7ea,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,129.16,129.16,69.16,72892002,Normal pregnancy,EPIC_PROD


In [ ]:
# unique encounter id
encounter2_ed.shape[0] ==  encounter2_ed['Id'].nunique()

True

In [24]:
add_encounters = encounter2_ed[~encounter2_ed['Id'].isin(encounters_ed['Id'])]
print(f'additional encounters from schema_change: {add_encounters.shape[0]}')

additional encounters from schema_change: 0


In [ ]:
common_encounters = encounter2_ed[encounter2_ed['Id'].isin(encounters_ed['Id'])]

#compare schema difference
enc_merge = encounters_ed.merge(common_encounters, how='inner', left_on=['Id'], right_on=['Id'], suffixes=('_enc', '_bat'))

,BASE_ENCOUNTER_COST_bat,BASE_ENCOUNTER_COST_enc,CODE_bat,CODE_enc,DESCRIPTION_bat,DESCRIPTION_enc,ENCOUNTERCLASS,ENCOUNTER_TYPE,Id,ORGANIZATION_bat,ORGANIZATION_enc,PATIENT_bat,PATIENT_enc,PAYER_COVERAGE_bat,PAYER_COVERAGE_enc,PAYER_bat,PAYER_enc,PROVIDER_bat,PROVIDER_enc,REASONCODE_bat,REASONCODE_enc,REASONDESCRIPTION_bat,REASONDESCRIPTION_enc,SOURCE_SYSTEM,START_bat,START_enc,STOP_bat,STOP_enc,TOTAL_CLAIM_COST_bat,TOTAL_CLAIM_COST_enc
0,129.16,129.16,50849002,50849002,Emergency room admission (procedure),Emergency room admission (procedure),emergency,emergency,2143f391-595c-4527-9b3d-68df74e28d26,465de31f-3098-365c-af70-48a071e1f5aa,465de31f-3098-365c-af70-48a071e1f5aa,53d371cc-eb68-4740-a68c-39707c6cb519,53d371cc-eb68-4740-a68c-39707c6cb519,89.16,89.16,7caa7254-5050-3b5e-9eae-bd5ea30e809c,7caa7254-5050-3b5e-9eae-bd5ea30e809c,0a8a9359-7b33-3256-a068-b5a7d18ebe4b,0a8a9359-7b33-3256-a068-b5a7d18ebe4b,<NA>,NaN,NaN,NaN,EPIC_PROD,2012-04-30 17:12:03,2012-04-30 18:12:03+00:00,2012-04-30 19:31:03,2012-04-30 20:31:03+00:00,129.16,129.16
1,129.16,129.16,50849002,50849002,Emergency Encounter,Emergency Encounter,emergency,emergency,27b3acba-4394-409a-a022-3b79da7a8f84,d733d4a9-080d-3593-b910-2366e652b7ea,d733d4a9-080d-3593-b910-2366e652b7ea,a708bca4-8fc9-4567-8c22-8b13d44f4f45,a708bca4-8fc9-4567-8c22-8b13d44f4f45,89.16,89.16,7caa7254-5050-3b5e-9eae-bd5ea30e809c,7caa7254-5050-3b5e-9eae-bd5ea30e809c,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,<NA>,NaN,NaN,NaN,CERNER_LEGACY,2019-01-25 04:27:38,2019-01-25 04:27:38+00:00,2019-01-25 05:42:38,2019-01-25 05:42:38+00:00,129.16,129.16
2,129.16,129.16,183460006,183460006,Obstetric emergency hospital admission,Obstetric emergency hospital admission,emergency,emergency,9a1650bf-5c2e-472d-a8fd-eb264321167c,d733d4a9-080d-3593-b910-2366e652b7ea,d733d4a9-080d-3593-b910-2366e652b7ea,5bb2161d-1383-408b-a96d-344c0bd5f23a,5bb2161d-1383-408b-a96d-344c0bd5f23a,69.16,69.16,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,72892002,72892002.0,Normal pregnancy,Normal pregnancy,EPIC_PROD,2018-12-11 06:15:56,2018-12-11 06:15:56+00:00,2018-12-11 07:45:56,2018-12-11 07:45:56+00:00,129.16,129.16
3,129.16,129.16,50849002,50849002,Emergency room admission (procedure),Emergency room admission (procedure),emergency,emergency,267f25e6-82fc-4966-9ce7-ab1632a68523,3d10019f-c88e-3de5-9916-6107b9c0263d,3d10019f-c88e-3de5-9916-6107b9c0263d,875d9a6a-1c7e-44a9-96a0-b4ba92e40841,875d9a6a-1c7e-44a9-96a0-b4ba92e40841,64.16,64.16,4d71f845-a6a9-3c39-b242-14d25ef86a8d,4d71f845-a6a9-3c39-b242-14d25ef86a8d,4b04cd2f-3f27-35bc-8069-f4ca6339529f,4b04cd2f-3f27-35bc-8069-f4ca6339529f,<NA>,NaN,NaN,NaN,EPIC_PROD,2012-07-14 03:44:13,2012-07-14 04:44:13+00:00,2012-07-14 04:44:13,2012-07-14 05:44:13+00:00,129.16,129.16


In [37]:
print(f'Almost identical data apart from start and stop date field')
enc_merge[sorted(enc_merge.columns)].head(4)

Almost identical data apart from start and stop date field


,BASE_ENCOUNTER_COST_bat,BASE_ENCOUNTER_COST_enc,CODE_bat,CODE_enc,DESCRIPTION_bat,DESCRIPTION_enc,ENCOUNTERCLASS,ENCOUNTER_TYPE,Id,ORGANIZATION_bat,ORGANIZATION_enc,PATIENT_bat,PATIENT_enc,PAYER_COVERAGE_bat,PAYER_COVERAGE_enc,PAYER_bat,PAYER_enc,PROVIDER_bat,PROVIDER_enc,REASONCODE_bat,REASONCODE_enc,REASONDESCRIPTION_bat,REASONDESCRIPTION_enc,SOURCE_SYSTEM,START_bat,START_enc,STOP_bat,STOP_enc,TOTAL_CLAIM_COST_bat,TOTAL_CLAIM_COST_enc
0,129.16,129.16,50849002,50849002,Emergency room admission (procedure),Emergency room admission (procedure),emergency,emergency,2143f391-595c-4527-9b3d-68df74e28d26,465de31f-3098-365c-af70-48a071e1f5aa,465de31f-3098-365c-af70-48a071e1f5aa,53d371cc-eb68-4740-a68c-39707c6cb519,53d371cc-eb68-4740-a68c-39707c6cb519,89.16,89.16,7caa7254-5050-3b5e-9eae-bd5ea30e809c,7caa7254-5050-3b5e-9eae-bd5ea30e809c,0a8a9359-7b33-3256-a068-b5a7d18ebe4b,0a8a9359-7b33-3256-a068-b5a7d18ebe4b,<NA>,NaN,NaN,NaN,EPIC_PROD,2012-04-30 17:12:03,2012-04-30 18:12:03+00:00,2012-04-30 19:31:03,2012-04-30 20:31:03+00:00,129.16,129.16
1,129.16,129.16,50849002,50849002,Emergency Encounter,Emergency Encounter,emergency,emergency,27b3acba-4394-409a-a022-3b79da7a8f84,d733d4a9-080d-3593-b910-2366e652b7ea,d733d4a9-080d-3593-b910-2366e652b7ea,a708bca4-8fc9-4567-8c22-8b13d44f4f45,a708bca4-8fc9-4567-8c22-8b13d44f4f45,89.16,89.16,7caa7254-5050-3b5e-9eae-bd5ea30e809c,7caa7254-5050-3b5e-9eae-bd5ea30e809c,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,<NA>,NaN,NaN,NaN,CERNER_LEGACY,2019-01-25 04:27:38,2019-01-25 04:27:38+00:00,2019-01-25 05:42:38,2019-01-25 05:42:38+00:00,129.16,129.16
2,129.16,129.16,183460006,183460006,Obstetric emergency hospital admission,Obstetric emergency hospital admission,emergency,emergency,9a1650bf-5c2e-472d-a8fd-eb264321167c,d733d4a9-080d-3593-b910-2366e652b7ea,d733d4a9-080d-3593-b910-2366e652b7ea,5bb2161d-1383-408b-a96d-344c0bd5f23a,5bb2161d-1383-408b-a96d-344c0bd5f23a,69.16,69.16,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,8cddbb1b-7e2c-3bf0-957e-bc62c160cfc5,72892002,72892002.0,Normal pregnancy,Normal pregnancy,EPIC_PROD,2018-12-11 06:15:56,2018-12-11 06:15:56+00:00,2018-12-11 07:45:56,2018-12-11 07:45:56+00:00,129.16,129.16
3,129.16,129.16,50849002,50849002,Emergency room admission (procedure),Emergency room admission (procedure),emergency,emergency,267f25e6-82fc-4966-9ce7-ab1632a68523,3d10019f-c88e-3de5-9916-6107b9c0263d,3d10019f-c88e-3de5-9916-6107b9c0263d,875d9a6a-1c7e-44a9-96a0-b4ba92e40841,875d9a6a-1c7e-44a9-96a0-b4ba92e40841,64.16,64.16,4d71f845-a6a9-3c39-b242-14d25ef86a8d,4d71f845-a6a9-3c39-b242-14d25ef86a8d,4b04cd2f-3f27-35bc-8069-f4ca6339529f,4b04cd2f-3f27-35bc-8069-f4ca6339529f,<NA>,NaN,NaN,NaN,EPIC_PROD,2012-07-14 03:44:13,2012-07-14 04:44:13+00:00,2012-07-14 04:44:13,2012-07-14 05:44:13+00:00,129.16,129.16


UUIDs appears to share the master index given the degree of identical data

the timestamps in the encounter table are UTC. 

In the encounter_schema_change dataset, the start and stop timestamp appears to be UTC-1 during summer time, and back to UTC+0 during winter time. Not possible with any timezones, will need further investigation.

In [38]:
enc_merge[['Id', 'START_enc', 'START_bat', 'STOP_enc', 'STOP_bat']]

,Id,START_enc,START_bat,STOP_enc,STOP_bat
0,2143f391-595c-4527-9b3d-68df74e28d26,2012-04-30 18:12:03+00:00,2012-04-30 17:12:03,2012-04-30 20:31:03+00:00,2012-04-30 19:31:03
1,27b3acba-4394-409a-a022-3b79da7a8f84,2019-01-25 04:27:38+00:00,2019-01-25 04:27:38,2019-01-25 05:42:38+00:00,2019-01-25 05:42:38
2,9a1650bf-5c2e-472d-a8fd-eb264321167c,2018-12-11 06:15:56+00:00,2018-12-11 06:15:56,2018-12-11 07:45:56+00:00,2018-12-11 07:45:56
3,267f25e6-82fc-4966-9ce7-ab1632a68523,2012-07-14 04:44:13+00:00,2012-07-14 03:44:13,2012-07-14 05:44:13+00:00,2012-07-14 04:44:13
4,3144dfd1-037f-416c-bf3b-96d02668203a,2011-09-16 00:16:26+00:00,2011-09-15 23:16:26,2011-09-16 01:16:26+00:00,2011-09-16 00:16:26
5,aae75d09-c955-45b3-b050-4fcda58a84a0,1999-09-14 17:27:52+00:00,1999-09-14 16:27:52,1999-09-14 18:57:52+00:00,1999-09-14 17:57:52
